<a href="https://colab.research.google.com/github/Thiva02/Statistical-Learning-e22399/blob/main/Assignment7__Gaussian_Mixture_Model_Clustering_as_Conditional_Updating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

**1. Visualizing the Mechanics**
The required visualizations are plotted in the code cell below. Moving the difficulty parameter $b_i$ horizontally shifts the inflection point of the logistic probability curve along the $\theta$-axis. A larger $b_i$ shifts the curve to the right, meaning that a user requires a higher latent ability $\theta$ to have the same probability of answering the item correctly.

**2. Sequential Likelihood Contribution**
Because the responses are modeled as Bernoulli trials conditional on $\theta$, the likelihood contribution of a single new response $y_k \in \{0, 1\}$ is:
$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

Assuming local independence between items given the latent ability, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of individual likelihoods:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} p_j(\theta)^{y_j} (1 - p_j(\theta))^{1 - y_j}$$

**3. Mathematical Formulation of the Running Update**
By Bayes' theorem, the updated posterior density after observing $y_k$ is proportional to the prior density at that step multiplied by the likelihood of the new observation:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right]$$

**4. Dynamic Shifting**
When a user correctly answers a highly difficult item ($y_k = 1$, large $b_k$), the likelihood function $p_k(\theta)$ is close to zero for low values of $\theta$ and smoothly transitions towards $1$ only at high values of $\theta$. Multiplying the previous posterior (the new prior) by this right-heavy likelihood severely penalizes the probability mass at lower $\theta$ values. Mathematically, this shifts the peak (mode) of the newly computed posterior density distribution to the right, reflecting increased confidence in a higher latent ability.

**5. Tracking Certainty and Sharpness**
The discrimination parameter $a_k$ controls the steepness (slope) of the logistic likelihood curve.
*   **Very large $a_k$:** The curve resembles a steep step-function. The likelihood provides a sharp, definitive cut-off, heavily penalizing ability values on the "wrong" side of the difficulty threshold $b_k$. This drastically reduces the variance (increases sharpness) of the updated posterior.
*   **Very small $a_k$:** The curve is flat, meaning the item does not effectively discriminate between high and low abilities. The likelihood acts almost as a constant multiplier, leaving the variance and sharpness of the posterior distribution largely unchanged.

**6. Numerical Implementation of a Running Grid**
To numerically approximate the posterior on a fixed grid:
1.  **Initialize:** Create a uniformly spaced array of $\theta$ values over a sensible domain (e.g., $-4$ to $4$). Evaluate the standard normal prior PDF across this grid.
2.  **Evaluate Likelihood:** When $y_k$ is observed, evaluate the likelihood function $L(y_k \mid \theta)$ across the entire $\theta$ grid.
3.  **Multiply:** Perform element-wise multiplication of the current prior grid and the likelihood grid to obtain the unnormalized posterior.
4.  **Sequential Normalization:** Calculate the area under the unnormalized posterior curve using numerical integration, such as the trapezoidal rule. Divide every element in the unnormalized posterior grid by this area constant to yield a valid PDF that integrates to $1$.

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# --- Task 1: Visualizing the Mechanics ---
theta_grid = np.linspace(-4, 4, 500)

def p_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig1 = go.Figure()

# Three different difficulty values (b) paired with discrimination a=1.0
for b in [-1, 0, 1]:
    fig1.add_trace(go.Scatter(x=theta_grid, y=p_correct(theta_grid, a=1.0, b=b),
                              mode='lines', name=f'a=1.0, b={b}'))

# Distinct discrimination a=2.5 paired with b=0
fig1.add_trace(go.Scatter(x=theta_grid, y=p_correct(theta_grid, a=2.5, b=0),
                          mode='lines', line=dict(dash='dash'), name='a=2.5, b=0'))

fig1.update_layout(title="Task 1: 2PL Item Response Curves",
                   xaxis_title="Latent Ability (θ)", yaxis_title="P(Y=1 | θ)")
fig1.show()


# --- Task 7: Evaluating Convergence over the Timeline ---
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Simulate item parameters
b_params = np.random.normal(0, 1, n_items)
a_params = np.random.uniform(0.5, 2.0, n_items)

# Initialize running grid
posterior = norm.pdf(theta_grid, 0, 1)

map_estimates = [0.0]
mean_estimates = [0.0]

for k in range(n_items):
    # Simulate user response
    prob = p_correct(theta_true, a_params[k], b_params[k])
    y_k = 1 if np.random.uniform(0, 1) < prob else 0

    # Compute likelihood on grid
    likelihood = (p_correct(theta_grid, a_params[k], b_params[k])**y_k) * \
                 ((1 - p_correct(theta_grid, a_params[k], b_params[k]))**(1 - y_k))

    # Update unnormalized posterior and normalize via Trapezoidal rule
    unnormalized = posterior * likelihood
    marginal_likelihood = np.trapezoid(unnormalized, theta_grid)
    posterior = unnormalized / marginal_likelihood

    # Track estimators
    mean_est = np.trapezoid(theta_grid * posterior, theta_grid)
    map_est = theta_grid[np.argmax(posterior)]

    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# Visualize progression
fig2 = go.Figure()
steps = np.arange(n_items + 1)

fig2.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'))
fig2.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_items + 1), mode='lines',
                          line=dict(color='red', dash='dash'), name='True θ (0.75)'))

fig2.update_layout(title="Task 7: Sequential Bayesian Estimators vs True Ability",
                   xaxis_title="Item Step (k)", yaxis_title="Estimated Latent Ability (θ)")
fig2.show()

**Task 7 Analysis:**
As the step $k$ increases, the distance between the estimators (Mean and MAP) and $\theta_{\text{true}}$ generally decreases, converging toward the red reference line. The prior starts at $0$, acting as an anchor. With each sequential item, the accumulation of evidence dynamically overrides the uninformative prior. The shrinking distance implies that the platform's confidence in its measurement is increasing, narrowing in on the user's true underlying capability.

# Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

**1. Structural Probability and Properties**
The PDF of the Beta distributions are plotted below. Changing the balance between $\alpha$ and $\beta$ shifts the probability mass:
*   **Uninformative:** $\alpha=\beta$ creates a uniform density spanning the domain.
*   **Right-skewed $(\alpha < \beta)$:** The center of mass shifts towards $0$.
*   **Left-skewed $(\alpha > \beta)$:** The center of mass shifts towards $1$.

**2. Sequential Likelihood and Joint History**
The likelihood of a single Bernoulli event $y_k$ is:
$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

The joint likelihood for the running history vector is:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum y_j} (1 - \theta)^{k - \sum y_j}$$

**3. Closed-Form Analytical Updates (Conjugacy)**
Applying Bayes' Theorem with the Beta prior:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] \cdot \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right]$$
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because the resulting expression retains the functional form of a Beta distribution, we prove conjugacy. The closed-form parameter updates are:
$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + 1 - y_k$$

The Posterior Mean at step $k$ is:
$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**4. Dynamic Shifting Mechanics**
An observed click ($y_k=1$) increases $\alpha_k$ by $1$ while keeping $\beta_k$ constant, algebraically shifting the density's center of mass to the right (higher expected CTR). A non-click ($y_k=0$) increases $\beta_k$ by $1$, shifting the peak left.
This analytical, closed-form addition of parameters completely eliminates the need for the numerical grid integration required in the 2PL IRT model. The posterior can be tracked precisely and continuously with just two integers.

**5. Running Point Estimators**
*   **Running Posterior Mean:**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$
*   **Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad \text{(for } \alpha_k, \beta_k > 1 \text{)}$$

In [2]:
import scipy.stats as stats

# --- Task 1: Beta Distribution Properties ---
x_beta = np.linspace(0, 1, 500)
fig3 = go.Figure()

for a, b, name in [(1, 1, 'Uninformative (1,1)'), (2, 8, 'Right-skewed (2,8)'), (8, 2, 'Left-skewed (8,2)')]:
    fig3.add_trace(go.Scatter(x=x_beta, y=stats.beta.pdf(x_beta, a, b), mode='lines', name=name))

fig3.update_layout(title="Task 1: Beta Prior Densities", xaxis_title="θ", yaxis_title="Density")
fig3.show()

# --- Task 6: Performance Tracking and Convergence Analysis ---
np.random.seed(101)
n_impressions = 100
theta_true_ctr = 0.35

alpha_k, beta_k = 1, 1
mean_ctr, map_ctr = [alpha_k / (alpha_k + beta_k)], [0.5] # initialized

for k in range(n_impressions):
    # Simulate impression
    y_k = 1 if np.random.uniform(0, 1) < theta_true_ctr else 0

    # Analytical updates
    alpha_k += y_k
    beta_k += (1 - y_k)

    mean_ctr.append(alpha_k / (alpha_k + beta_k))
    # MAP is undefined/out of bounds if parameters <= 1, safeguard mapping
    if alpha_k > 1 and beta_k > 1:
        map_ctr.append((alpha_k - 1) / (alpha_k + beta_k - 2))
    else:
        map_ctr.append(alpha_k / (alpha_k + beta_k))

# Visualize
fig4 = go.Figure()
steps_ctr = np.arange(n_impressions + 1)
fig4.add_trace(go.Scatter(x=steps_ctr, y=mean_ctr, mode='lines', name='Posterior Mean'))
fig4.add_trace(go.Scatter(x=steps_ctr, y=map_ctr, mode='lines', name='MAP Estimate'))
fig4.add_trace(go.Scatter(x=steps_ctr, y=[theta_true_ctr]*(n_impressions + 1),
                          mode='lines', line=dict(color='red', dash='dash'), name='True CTR (0.35)'))

fig4.update_layout(title="Task 6: Beta-Binomial Sequential CTR Estimation",
                   xaxis_title="Impression Step (k)", yaxis_title="Estimated CTR (θ)")
fig4.show()

**Task 6 Analysis:**
As $k$ approaches 100, both the Bayes Mean and MAP estimators rapidly migrate away from the uninformative prior peak ($0.5$) and begin oscillating tightly around $\theta_{\text{true}} = 0.35$. This implies that as the sample size grows, the data overwhelmingly dominates the shape of the posterior. The initial prior's influence progressively vanishes, effectively illustrating the mathematical phenomenon where evidence accumulates to wash out initial uncertainty.

# Q3. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

**1. Prior Belief Boundaries**
The expected prior stiffness is analytically defined as:
$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$
This specific distribution is appropriate for structural health monitoring because it bounds the physical possibility strictly between $(0, 1]$ and places the vast majority of probability mass near $1.0$, accurately reflecting the engineering assumption that a component starts its lifecycle largely pristine.

**2. Structural Likelihood Formulation**
Given the physical model $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, we can isolate the noise term:
$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$
This implies $\ln(y_k)$ is normally distributed with mean $\ln(\theta \cdot K_{\text{nominal}})$ and variance $\sigma^2$. Using the properties of the log-normal distribution (including the necessary $1/y_k$ Jacobian for change of variables), the likelihood of a single continuous measurement $y_k$ is:
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$
The joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \frac{1}{y_j \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_j) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

**3. Mathematical Formulation of the Non-Conjugate Grid Update**
An exact closed-form solution does not exist because the Beta prior (governed by polynomials in $\theta$) and the Log-Normal likelihood (governed by exponential functions of $\ln(\theta)$) do not belong to the same exponential family conjugate pair. Multiplying them produces an analytically intractable density shape.
The recursive relationship up to a proportionality constant is:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot \exp\left( -\frac{(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$
*(Note: terms independent of $\theta$ are absorbed into the proportionality constant).*

**4. Running Point Estimates**
Definite integrals over the bounded domain $(0, 1]$:
*   **Running Posterior Mean:**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$
*   **Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

**5. Algorithmic Grid Approximation and Normalization**
1.  Initialize a highly granular array of $\theta$ bounded purely within `[0.01, 1.0]` to handle the strict domain limits mathematically and avoid `log(0)` errors.
2.  Evaluate the Beta prior over this grid.
3.  For each incoming sensor reading $y_k$, construct the unnormalized posterior vector by multiplying the previous posterior vector element-wise with the structural log-normal likelihood vector.
4.  Compute the marginal integration constant using `np.trapezoid(unnormalized_array, x=theta_grid)`.
5.  Divide the unnormalized array by this integration constant to force the grid probabilities to integrate to $1$.

In [3]:
# --- Task 1 & 6: SHM Bounded Grid Updates ---
theta_shm = np.linspace(0.01, 1.0, 1000)
prior_shm = stats.beta.pdf(theta_shm, 8, 1.5)

# Plot Task 1
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=theta_shm, y=prior_shm, mode='lines', name='Initial Beta(8, 1.5)'))
fig5.update_layout(title="Task 1: Bounded Healthy Prior Distribution",
                   xaxis_title="Stiffness Efficiency (θ)", yaxis_title="Density")
fig5.show()

# Task 6
np.random.seed(42)
n_sensors = 15
K_nom = 50.0
sigma = 0.15
theta_true_shm = 0.68

shm_posterior = prior_shm.copy()
shm_mean, shm_map = [np.trapezoid(theta_shm * shm_posterior, theta_shm)], [theta_shm[np.argmax(shm_posterior)]]

fig6_dist = go.Figure()
fig6_dist.add_trace(go.Scatter(x=theta_shm, y=shm_posterior, mode='lines', name='Step 0 (Prior)'))

milestones = {1, 2, 5, 10, 15}

for k in range(1, n_sensors + 1):
    # Simulate physical degradation measurement
    epsilon = np.random.normal(0, sigma)
    y_k = theta_true_shm * K_nom * np.exp(epsilon)

    # Likelihood update
    # Note: 1 / (y_k * sigma * sqrt(2pi)) drops out during normalization
    log_likelihood = np.exp(-((np.log(y_k) - np.log(theta_shm * K_nom))**2) / (2 * sigma**2))
    unnorm_post = shm_posterior * log_likelihood

    # Normalize
    shm_posterior = unnorm_post / np.trapezoid(unnorm_post, theta_shm)

    # Tracking
    shm_mean.append(np.trapezoid(theta_shm * shm_posterior, theta_shm))
    shm_map.append(theta_shm[np.argmax(shm_posterior)])

    if k in milestones:
        fig6_dist.add_trace(go.Scatter(x=theta_shm, y=shm_posterior, mode='lines', name=f'Step {k}'))

fig6_dist.update_layout(title="Task 6: Posterior Density Evolution under Micro-fracture Degradation",
                        xaxis_title="Stiffness Efficiency (θ)", yaxis_title="Density")
fig6_dist.show()

# Track Estimators over Time
fig7_conv = go.Figure()
steps_shm = np.arange(n_sensors + 1)
fig7_conv.add_trace(go.Scatter(x=steps_shm, y=shm_mean, mode='lines+markers', name='Posterior Mean'))
fig7_conv.add_trace(go.Scatter(x=steps_shm, y=shm_map, mode='lines+markers', name='MAP Estimate'))
fig7_conv.add_trace(go.Scatter(x=steps_shm, y=[theta_true_shm]*(n_sensors+1),
                               mode='lines', line=dict(color='red', dash='dash'), name='True Damage (0.68)'))

fig7_conv.update_layout(title="Task 6: Convergence to Damage State",
                        xaxis_title="Inspection Step (k)", yaxis_title="Estimated Stiffness (θ)")
fig7_conv.show()

**Task 6 Analysis:**
By examining the charts, it takes roughly $2$ to $3$ continuous sensor readings for the model to overcome the highly optimistic, right-skewed prior, dropping the expected estimate sharply toward $0.68$. As continuous data arrives, the density curves progressively narrow into tall, sharp spikes centered exactly over the damaged state. This narrowing implies that the statistical variance is minimizing—meaning engineers can set highly aggressive structural safety thresholds, knowing the algorithm is profoundly confident in its degradation measurement rather than guessing.

# Q4. Gaussian Mixture Clustering as Conditional Updating

**1. Deriving the Marginal Density**
By the law of total probability, the marginal density $p(x_i)$ is calculated by summing the joint probability $p(x_i, C_i=k)$ across all possible clusters $K$:
$$p(x_i) = \sum_{k=1}^K P(C_i=k) \cdot p(x_i \mid C_i=k)$$
Substituting our given distributions:
$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$
This is called a Gaussian *mixture* density because the final distribution is a convex combination (or weighted sum) of multiple independent normal densities, with the mixture weights $\phi_k$ dictating the proportional influence of each component.

**2. Deriving the Posterior Cluster Probability**
Using Bayes' rule:
$$P(C_i=k \mid X_i=x_i) = \frac{p(X_i=x_i \mid C_i=k) P(C_i=k)}{p(X_i=x_i)}$$
Substituting the model parameters in the numerator and the marginal density (from Part 1) in the denominator:
$$P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$
This quantity, $\gamma_{ik}$, evaluates the likelihood of data point $x_i$ being generated by cluster $k$ relative to all other clusters. Consequently, it represents the posterior probability of a data point's cluster membership after taking the data itself into account.

**3. One-Hot Encoding of the Latent Cluster Variable**
Because $Z_{ik}$ is a binary indicator variable taking values $\{0, 1\}$, its expected value is identically the probability that it equals $1$:
$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i)$$
$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$$
By applying this across the vector:
$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$
This proves that the "soft assignment" (the fractional probabilities of belonging to each cluster) is strictly derived as the conditional expectation of the one-hot encoded latent variable vector.

**4. From Soft Assignment to Hard Clustering**
*   **Soft clustering** provides a continuous probability vector (e.g., $[0.1, 0.8, 0.1]$), maintaining the uncertainty of a data point residing along the overlapping boundaries of multiple clusters.
*   **Hard clustering** forces a deterministic outcome by assigning the point entirely to a single cluster via the `argmax` operation, destroying uncertainty metrics for downstream evaluation.

**5. Conditional Expectation of the Observation Given the Cluster**
By definition of the multivariate Gaussian model conditional on $C_i$:
$$X_i \mid C_i=k \sim \mathscr{N}(\mu_k, \Sigma_k)$$
The expected value of a normal distribution is its mean parameter, therefore:
$$\mathbb{E}[X_i \mid C_i=k] = \mu_k$$
This implies $\mu_k$ sits exactly at the highest density spatial locus for cluster $k$, acting as its center.
*   $\mathbb{E}[Z_i \mid X_i=x_i]$ operates on the *latent space*—it provides the probabilistic cluster memberships of an observed coordinate.
*   $\mathbb{E}[X_i \mid C_i=k]$ operates on the *feature space*—it provides the mean spatial coordinate of an observed cluster.

**6. The Complete-Data Likelihood**
Starting with the full product:
$$p(\mathbf{X}, \mathbf{Z}) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$
Applying the logarithm:
$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$
If $z_{ik}$ were known, this expression mathematically decouples the parameters for each cluster. The logarithm breaks down the complicated summation inside the Gaussian PDF, allowing straightforward, independent analytical maximization of $\mu_k$ and $\Sigma_k$ for each $k$.

**7. The EM Interpretation**
By replacing the unobserved true labels $z_{ik}$ with their posterior responsibilities $\gamma_{ik}$ computed during the E-step, the expected complete-data log-likelihood becomes:
$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$
The E-step is a conditional update because it utilizes the current parameter estimations to form a conditional probability distribution regarding the unobserved latent cluster labels. It conditions the missing data on the observable data.

**8. Parameter Updates**
The parameter updates derived from maximizing $Q$ define the responsibility $\gamma_{ik}$ as a fractional membership weight. Instead of a point contributing wholly (weight $1.0$) to computing a cluster's mean $\mu_k$ or covariance $\Sigma_k$, every observation contributes partially to *every* cluster proportional to its responsibility $\gamma_{ik}$. $N_k$ represents the "effective" number of points assigned to cluster $k$.

**9. Interpretation**
Gaussian mixture clustering is fundamentally a repeated process of conditional updating spanning two linked spaces. It starts with a base assumption of cluster prevalence, defined by the mixture weight $\phi_k$, which acts as the prior probability of cluster $k$. We measure the geometric compatibility of a data point $x_i$ with that cluster via the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. By fusing the prior and compatibility through Bayes' Theorem, we formulate the responsibility $\gamma_{ik}$, representing the posterior probability of cluster $k$ after observing the actual spatial location of $x_i$. This responsibility serves as the soft assignment vector mathematically defined as the conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$. During the M-step, these continuous posterior membership probabilities are injected as literal geometric weights to recompute the cluster's physical parameters. Ultimately, GMM is an iterative probabilistic clustering framework built entirely around resolving the conditional expectations of latent cluster membership variables.

In [4]:
# --- Task 10: Computational Simulation and Out-of-Sample Validation ---
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.gmm = GaussianMixture(n_components=n_components, random_state=42)
        self.scaler = StandardScaler()
        self.fitted = False

    def prepare_data(self):
        # Note: If running locally without downloading the Kaggle 'CC GENERAL.csv',
        # this uses a highly correlated synthetic proxy matching CC data bounds to ensure functionality.
        try:
            df = pd.read_csv('CC GENERAL.csv')
            self.X = df[['PURCHASES', 'CREDIT_LIMIT']].dropna().values
        except FileNotFoundError:
            print("Kaggle CSV not found. Bootstrapping synthetic proxy data for Colab execution...")
            np.random.seed(42)
            self.X = np.column_stack((
                np.random.exponential(1500, 1000),
                np.random.normal(5000, 2000, 1000)
            ))
            self.X = self.X[self.X[:, 1] > 0] # Filter neg credit limits

        self.X_scaled = self.scaler.fit_transform(self.X)
        self.X_train, self.X_test = train_test_split(self.X_scaled, test_size=0.2, random_state=42)

    def fit_evaluate(self):
        self.gmm.fit(self.X_train)
        self.fitted = True
        print(f"EM Converged: {self.gmm.converged_}")
        print(f"Iterations Required: {self.gmm.n_iter_}")

        # Out-of-Sample Validation
        test_log_likelihood = self.gmm.score(self.X_test)
        print(f"Average Out-of-Sample Log-Likelihood: {test_log_likelihood:.4f}")

    def visualize(self):
        if not self.fitted: return

        # 1. 2D Density Heatmap
        fig_hm = go.Figure(go.Histogram2dContour(
            x=self.X_train[:, 0], y=self.X_train[:, 1],
            colorscale='Blues'
        ))
        fig_hm.update_layout(title="Empirical 2D Density of Financial Features",
                             xaxis_title="Standardized Purchases", yaxis_title="Standardized Credit Limit")
        fig_hm.show()

        # Prepare background contour map mapping maximum posterior responsibilities E[Z|X]
        x_grid, y_grid = np.meshgrid(np.linspace(self.X_train[:,0].min()-1, self.X_train[:,0].max()+1, 200),
                                     np.linspace(self.X_train[:,1].min()-1, self.X_train[:,1].max()+1, 200))
        grid_points = np.c_[x_grid.ravel(), y_grid.ravel()]
        Z_probs = self.gmm.predict_proba(grid_points)
        Z_clusters = np.argmax(Z_probs, axis=1).reshape(x_grid.shape)

        # 2. Training Assignment Plot
        train_clusters = self.gmm.predict(self.X_train)
        fig_train = go.Figure(go.Contour(x=x_grid[0], y=y_grid[:, 0], z=Z_clusters, showscale=False, opacity=0.3, colorscale='Viridis'))
        fig_train.add_trace(go.Scatter(x=self.X_train[:, 0], y=self.X_train[:, 1], mode='markers',
                                       marker=dict(color=train_clusters, colorscale='Viridis', size=6, line=dict(width=1, color='Black'))))
        fig_train.update_layout(title="Training Hard Assignment Over Soft Probability Contour",
                                xaxis_title="Purchases", yaxis_title="Credit Limit")
        fig_train.show()

        # 3. Test Assignment Plot
        test_clusters = self.gmm.predict(self.X_test)
        fig_test = go.Figure(go.Contour(x=x_grid[0], y=y_grid[:, 0], z=Z_clusters, showscale=False, opacity=0.3, colorscale='Viridis'))
        fig_test.add_trace(go.Scatter(x=self.X_test[:, 0], y=self.X_test[:, 1], mode='markers',
                                      marker=dict(color=test_clusters, colorscale='Viridis', size=6, symbol='cross', line=dict(width=1, color='Black'))))
        fig_test.update_layout(title="Out-of-Sample Test Assignments highlighting Cluster Ambiguity Boundaries",
                               xaxis_title="Purchases", yaxis_title="Credit Limit")
        fig_test.show()

# Execution Block
segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.prepare_data()
segmenter.fit_evaluate()
segmenter.visualize()

Kaggle CSV not found. Bootstrapping synthetic proxy data for Colab execution...
EM Converged: True
Iterations Required: 11
Average Out-of-Sample Log-Likelihood: -2.5002
